# 🦵 Abnormality Detection from one knee MRI — EDA

**Study-level structure, report coverage, series geometry, and the two gauges that will matter once a model exists.**

This EDA is built to answer one question in each section: *what does the imaging model actually get to work with, and where does the supervision run thin?* It follows the same discipline a training pipeline for this data needs downstream — every diagnostic here either informs a modeling decision or rules one out.

| Section | Question it answers |
|---|---|
| 1 | What files exist, and what do the twelve targets look like where they're annotated? |
| 2 | How much of the corpus has an annotation vs. only a report? |
| 3 | What does the report text look like — language, length, missingness? |
| 4 | What does `train_series.csv` say about plane and acquisition coverage? |
| 5 | What do the DICOM headers themselves say — pixel spacing, slice thickness, laterality? |
| 6 | Do the twelve findings co-occur, or are they close to independent? |
| 7 | Summary — what this means for splitting, sampling, and slot design |

Two numbers get carried through the whole notebook rather than computed once and forgotten: **how many studies actually have a ground-truth label** (small — this drives every interval), and **how many have only free text** (large — this is what a report-derived label strategy is for).


## 1. Setup

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["pydicom", "networkx"]:
    try:
        __import__(pkg)
    except ImportError:
        pip_install(pkg)

import os, re, time, gc, hashlib
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

import pydicom

T0 = time.time()
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

SEED = 2026
np.random.seed(SEED)

# ------------------------------------------------------------------ icefire theme
# Same palette across every chart in this notebook, so a colour always means the same
# thing: cold end = low / normal / absent, warm end = high / abnormal / present.
ICEFIRE  = sns.color_palette("icefire", 12)
ICE      = ICEFIRE[2]
FIRE     = ICEFIRE[9]
MIDLINE  = "#39414d"
CMAP     = "icefire"
CMAP_SEQ = sns.blend_palette(["#eef1f6", ICE, FIRE], as_cmap=True)

sns.set_theme(style="white", palette=ICEFIRE, font_scale=0.95)
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c8cdd4", "axes.labelcolor": MIDLINE,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlepad": 10,
    "text.color": MIDLINE, "xtick.color": MIDLINE, "ytick.color": MIDLINE,
    "grid.color": "#e8eaee", "axes.grid": True, "axes.grid.axis": "y",
    "legend.frameon": False, "figure.facecolor": "white",
})

def grad(n, reverse=False):
    """n colours along the icefire ramp, skipping its near-black centre."""
    cm = plt.get_cmap(CMAP)
    cold = np.linspace(0.02, 0.28, n - n // 2)
    warm = np.linspace(0.78, 0.99, n // 2)
    cols = [cm(p) for p in np.concatenate([cold, warm])][:n]
    return cols[::-1] if reverse else cols

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
log("setup complete")


## 2. Load & Inventory

Same resolution logic a training script would use: try the standard mount points, fall back to a scan, and fail loudly rather than guessing a path.


In [ ]:
def find_root():
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file():
            return c
    for depth1 in sorted(p for p in Path("/kaggle/input").iterdir() if p.is_dir()):
        for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
            if (cand / "test.csv").is_file():
                return cand
    raise FileNotFoundError("competition mount not found — check /kaggle/input")

ROOT = find_root()
log(f"input root: {ROOT}")

train_df = pd.read_csv(ROOT / "train.csv")
train_series = pd.read_csv(ROOT / "train_series.csv")
test_df = pd.read_csv(ROOT / "test.csv")
test_series = pd.read_csv(ROOT / "test_series.csv")
sample_sub = pd.read_csv(ROOT / "sample_submission.csv")

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]
TARGETS = [t for t in TARGETS if t in train_df.columns]  # tolerate a renamed column

print(f"train.csv       {train_df.shape}")
print(f"train_series.csv{train_series.shape}")
print(f"test.csv        {test_df.shape}")
print(f"test_series.csv {test_series.shape}")
print(f"sample_submission {sample_sub.shape}")
print(f"\ntargets found in train.csv: {TARGETS}")


In [ ]:
display(train_df.head(3))
display(train_series.head(3))


## 3. What the twelve targets look like

Two populations sit inside `train.csv`: studies with a per-condition **annotation** (the twelve target columns filled in), and studies with only a **report** — the free-text field a radiologist wrote. Everything downstream depends on knowing which is which, because the annotated set is the only one an AUC can be honestly measured against, and it is small.


In [ ]:
annotated_mask = train_df[TARGETS].notna().all(axis=1)
n_annot = int(annotated_mask.sum())
n_total = len(train_df)

print(f"studies with a full annotation: {n_annot} / {n_total}  ({n_annot/n_total:.1%})")
print(f"studies with report only:       {n_total - n_annot} / {n_total}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Annotated", "Report only"], [n_annot, n_total - n_annot],
       color=[FIRE, ICE], edgecolor="white", width=.55)
for i, v in enumerate([n_annot, n_total - n_annot]):
    ax.text(i, v + n_total * 0.01, f"{v}\n({v/n_total:.1%})", ha="center", fontsize=10)
ax.set_title("Annotated vs. report-only studies")
ax.set_ylim(0, n_total * 1.15)
plt.tight_layout()
plt.show()


In [ ]:
gold = train_df.loc[annotated_mask, TARGETS].astype(float)

# Positive rate per target, annotated subset only — this is the population an AUC will
# actually be measured against downstream, so it is worth seeing on its own before any
# report-derived signal enters the picture.
pos_rate = (gold > 0.5).mean().sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(pos_rate.index, pos_rate.values * 100, color=grad(len(pos_rate)), edgecolor="white", height=.7)
for i, v in enumerate(pos_rate.values * 100):
    ax.text(v + 1, i, f"{v:.0f}%  (n={int((gold[pos_rate.index[i]]>0.5).sum())})",
            va="center", fontsize=9)
ax.set_xlim(0, max(pos_rate.values * 100) * 1.35)
ax.set_title(f"Positive rate on the annotated subset (n={n_annot})")
ax.set_xlabel("% positive")
plt.tight_layout()
plt.show()


**Why this matters before any model exists:** with only a few dozen positives for the rarest targets, the annotated subset alone cannot arbitrate small differences — the same Hanley–McNeil argument that governs a report-derived label extractor governs any EDA claim made on this slice too. The interval below is drawn for each target's positive rate using the normal approximation to a binomial proportion, so it's visible before anyone is tempted to read too much into a 3-percentage-point gap.


In [ ]:
def wilson_interval(p, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    adj = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)
    return ((centre - adj) / denom, (centre + adj) / denom)

rows = []
for t in TARGETS:
    p = (gold[t] > 0.5).mean()
    n = gold[t].notna().sum()
    lo, hi = wilson_interval(p, n)
    rows.append({"target": t, "pos_rate": p, "n": int(n), "lo95": lo, "hi95": hi})
rate_df = pd.DataFrame(rows).sort_values("pos_rate")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(rate_df.target, rate_df.pos_rate * 100, color=grad(len(rate_df)), edgecolor="white", height=.6)
ax.errorbar(rate_df.pos_rate * 100, np.arange(len(rate_df)),
            xerr=[(rate_df.pos_rate - rate_df.lo95) * 100, (rate_df.hi95 - rate_df.pos_rate) * 100],
            fmt="none", ecolor=MIDLINE, elinewidth=1.2, capsize=3)
ax.set_title("Positive rate with 95% Wilson interval — rare targets have wide bars")
ax.set_xlabel("% positive")
plt.tight_layout()
plt.show()
display(rate_df.round(3))


## 4. The reports

`train.csv` carries a `Report` field on (presumably) every study, `test.csv` does not — the schema itself rules out any model that reads text at inference. What the reports are used for here is purely diagnostic: language mix, length, and missingness, all of which bound how far a report-derived labeling strategy could reach before it's built.


In [ ]:
has_report = "Report" in train_df.columns
print("Report column present:", has_report)

if has_report:
    report_len = train_df["Report"].fillna("").str.len()
    empty_reports = (train_df["Report"].isna() | (train_df["Report"].str.strip() == "")).mean()
    print(f"empty/missing reports: {empty_reports:.1%}")

    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.histplot(report_len[report_len > 0], bins=40, color=ICE, ax=ax)
    ax.axvline(report_len.median(), color=FIRE, ls="--", lw=1.5,
               label=f"median = {report_len.median():.0f} chars")
    ax.set_title("Report length distribution (characters)")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
if has_report:
    # Cheap language signature — Unicode block share, not a language classifier. Enough
    # to see the multilingual mix this corpus reportedly has, without pulling in a model.
    def script_signature(s):
        if not isinstance(s, str) or not s:
            return "empty"
        counts = Counter()
        for ch in s:
            o = ord(ch)
            if 0x0370 <= o <= 0x03FF:
                counts["greek"] += 1
            elif 0x0400 <= o <= 0x04FF:
                counts["cyrillic"] += 1
            elif ch.isalpha():
                counts["latin_or_other"] += 1
        if not counts:
            return "no_letters"
        return counts.most_common(1)[0][0]

    sig = train_df["Report"].fillna("").apply(script_signature)
    vc = sig.value_counts()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(vc.index, vc.values, color=grad(len(vc)), edgecolor="white", width=.55)
    for i, v in enumerate(vc.values):
        ax.text(i, v + n_total * 0.005, f"{v}", ha="center", fontsize=9)
    ax.set_title("Report script signature (Unicode block — not a language ID)")
    ax.tick_params(axis="x", rotation=15)
    plt.tight_layout()
    plt.show()
    print("Note: the Latin-script bucket mixes English, Spanish, French, Dutch, German,")
    print("Turkish and Croatian — a real language ID needs stopword scoring, not Unicode alone.")


In [ ]:
if has_report:
    # Duplicate reports matter for splitting, not just for text stats: several studies
    # sharing one template report share one derived target vector if labels are extracted
    # from text, and a group like that has to stay inside one fold.
    norm_report = train_df["Report"].fillna("").str.lower().str.strip()
    dup_counts = norm_report[norm_report != ""].value_counts()
    dup_groups = dup_counts[dup_counts > 1]

    print(f"distinct non-empty report texts: {norm_report[norm_report!=''].nunique()}")
    print(f"report texts shared by 2+ studies: {len(dup_groups)}")
    print(f"studies sitting inside a shared-report group: {int(dup_groups.sum())} "
          f"({dup_groups.sum()/n_total:.1%} of the corpus)")

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(dup_groups.values, bins=range(2, min(int(dup_groups.max()) + 2, 30)),
                 color=FIRE, ax=ax)
    ax.set_title("Size of shared-report groups (template reports)")
    ax.set_xlabel("studies sharing one exact report text")
    plt.tight_layout()
    plt.show()


## 5. Series-level metadata plane and acquisition coverage

`train_series.csv` names the anatomical plane and two acquisition flags, `Fluid_Sensitive` and `Fat_Suppression`. These are the two axes a slot design (sagittal/coronal/axial × weighting × fat-sat) is built from, so their coverage — not just their marginal counts — is what matters.


In [ ]:
plane_col = "Anatomical_Plane" if "Anatomical_Plane" in train_series.columns else \
            next((c for c in train_series.columns if "plane" in c.lower()), None)
fluid_col = "Fluid_Sensitive" if "Fluid_Sensitive" in train_series.columns else None
fatsat_col = "Fat_Suppression" if "Fat_Suppression" in train_series.columns else None
print("plane column:", plane_col, "| fluid column:", fluid_col, "| fat-sat column:", fatsat_col)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, col, title in zip(axes, [plane_col, fluid_col, fatsat_col],
                           ["Anatomical plane", "Fluid-sensitive flag", "Fat-suppression flag"]):
    if col is None:
        ax.axis("off")
        continue
    vc = train_series[col].value_counts(dropna=False)
    ax.bar(vc.index.astype(str), vc.values, color=grad(len(vc)), edgecolor="white", width=.6)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=20)
    for i, v in enumerate(vc.values):
        ax.text(i, v + max(vc.values) * 0.02, str(v), ha="center", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
if plane_col and fluid_col and fatsat_col:
    # Slot coverage grid: plane x (fluid, fat-sat) — this is the same grid a slot design
    # in a model pipeline would be built on, so seeing where it's thin here is cheaper
    # than discovering it after the cache is built.
    combo = train_series.groupby([plane_col, fluid_col, fatsat_col]).size().reset_index(name="count")
    combo["combo"] = combo[fluid_col].astype(str) + " / FS=" + combo[fatsat_col].astype(str)
    pivot = combo.pivot_table(index=plane_col, columns="combo", values="count", fill_value=0)

    plt.figure(figsize=(10, 4.5))
    sns.heatmap(pivot, annot=True, fmt="g", cmap=CMAP_SEQ, linewidths=0.5, linecolor="white")
    plt.title("Series count by plane × (fluid-sensitive / fat-suppression)")
    plt.tight_layout()
    plt.show()


In [ ]:
# Series (and slices, once weighted by n_slices if available) per study — the shape
# a per-study cache would need to budget for.
series_per_study = train_series.groupby("StudyInstanceUID").size()

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(series_per_study, bins=range(int(series_per_study.min()), int(series_per_study.max()) + 2),
             color=ICE, ax=ax)
ax.axvline(series_per_study.median(), color=FIRE, ls="--", lw=1.5,
           label=f"median = {series_per_study.median():.0f}")
ax.set_title("Series per study")
ax.legend()
plt.tight_layout()
plt.show()
print(series_per_study.describe())


## 6. DICOM headers physical scale and laterality

Two things live in the header that don't live in either CSV: **pixel spacing** (how many millimetres a pixel covers — the reason a fixed-pixel resize scrambles anatomical scale across studies with different spacing) and **laterality** (which knee — present on some studies via the `Laterality`/`ImageLaterality` tag, recoverable on others via the sign of the patient x-coordinate). Both are sampled here on a subset of files; reading every DICOM header in the corpus is unnecessary for an EDA pass and expensive.


In [ ]:
HDR_TAGS = ["Rows", "Columns", "PixelSpacing", "SliceThickness",
            "Laterality", "ImageLaterality", "ImagePositionPatient",
            "RepetitionTime", "EchoTime"]

def probe_header(path):
    try:
        ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        row = {"path": str(path)}
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
        return row
    except Exception as e:
        return {"path": str(path), "err": str(e)[:100]}

train_series_dir = ROOT / "train_series"
if train_series_dir.is_dir():
    all_dcm = list(train_series_dir.rglob("*.dcm"))
    sample_files = list(np.random.choice(all_dcm, min(400, len(all_dcm)), replace=False)) if all_dcm else []
    log(f"found {len(all_dcm)} DICOM files under train_series/, sampling {len(sample_files)}")

    with ThreadPoolExecutor(max_workers=16) as pool:
        hdr_rows = list(pool.map(probe_header, sample_files))
    hdr_df = pd.DataFrame(hdr_rows)
    log(f"probed {len(hdr_df)} headers")
else:
    hdr_df = pd.DataFrame()
    log("no train_series/ directory found — skipping header probe")

hdr_df.head()


In [ ]:
if len(hdr_df) and "PixelSpacing" in hdr_df.columns:
    px = pd.to_numeric(hdr_df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
                        errors="coerce")
    st = pd.to_numeric(hdr_df.get("SliceThickness", pd.Series(dtype=object)), errors="coerce")
    rows = pd.to_numeric(hdr_df.get("Rows", pd.Series(dtype=object)), errors="coerce")
    cols = pd.to_numeric(hdr_df.get("Columns", pd.Series(dtype=object)), errors="coerce")

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    sns.histplot(px.dropna(), bins=25, color=ICE, ax=axes[0,0])
    axes[0,0].set_title("Pixel spacing (mm/pixel)")

    sns.histplot(st.dropna(), bins=25, color=FIRE, ax=axes[0,1])
    axes[0,1].set_title("Slice thickness (mm)")

    sns.histplot(rows.dropna(), bins=20, color=ICE, ax=axes[1,0])
    axes[1,0].set_title("Image rows (pixels)")

    sns.histplot(cols.dropna(), bins=20, color=FIRE, ax=axes[1,1])
    axes[1,1].set_title("Image columns (pixels)")
    plt.tight_layout()
    plt.show()

    if px.notna().sum() > 5:
        fov_mm = px * rows
        print(f"pixel spacing spans {px.min():.2f}–{px.max():.2f} mm/px "
              f"({px.max()/px.min():.1f}x range)")
        print(f"implied field of view spans roughly {fov_mm.min():.0f}–{fov_mm.max():.0f} mm")
        print("A fixed-pixel resize (e.g. every image to 224x224) would therefore show the")
        print("model a meniscus at a different physical scale in different studies for no")
        print("anatomical reason. A physical-scale crop (fixed mm, then resize) removes that.")


In [ ]:
if len(hdr_df) and "Laterality" in hdr_df.columns:
    # Same reasoning as a laterality fallback in a training pipeline: check the tag's
    # coverage, and whether the x-coordinate sign actually agrees with it where both
    # exist, before trusting the coordinate on studies where the tag is missing.
    def tag_side(row):
        for col in ["Laterality", "ImageLaterality"]:
            v = row.get(col)
            if isinstance(v, str) and v.strip().upper()[:1] in ("L", "R"):
                return v.strip().upper()[0]
        return None

    def position_side(row, min_offset=5.0):
        v = row.get("ImagePositionPatient")
        if not isinstance(v, str):
            return None
        try:
            x = float(v.split("|")[0])
        except Exception:
            return None
        if abs(x) < min_offset:
            return None
        return "R" if x < 0 else "L"   # LPS convention: right side sits at negative x

    hdr_df["tag_side"] = hdr_df.apply(tag_side, axis=1)
    hdr_df["pos_side"] = hdr_df.apply(position_side, axis=1)

    tag_coverage = hdr_df["tag_side"].notna().mean()
    both = hdr_df.dropna(subset=["tag_side", "pos_side"])
    agreement = (both["tag_side"] == both["pos_side"]).mean() if len(both) else np.nan

    print(f"Laterality tag present on {tag_coverage:.1%} of sampled headers")
    print(f"x-coordinate sign agrees with the tag on {agreement:.1%} of "
          f"{len(both)} comparable headers")

    fig, ax = plt.subplots(figsize=(6, 4))
    labels = ["Tag present", "Tag missing"]
    values = [tag_coverage, 1 - tag_coverage]
    ax.pie(values, labels=labels, colors=[FIRE, ICE], autopct="%1.0f%%",
           wedgeprops={"edgecolor": "white"})
    ax.set_title("Laterality tag coverage (sampled headers)")
    plt.tight_layout()
    plt.show()

    if np.isfinite(agreement):
        verdict = "high enough to trust as a fallback" if agreement >= 0.85 else "too low to trust blindly"
        print(f"\n-> x-sign agreement is {verdict} for filling the gap where the tag is absent.")


## 7. Do the twelve findings co-occur?

A knee report rarely names one thing in isolation — an ACL tear often comes with an effusion, osteoarthritis in one compartment often accompanies another. This matters for whether a per-diagnosis model can be trained fully independently, or whether the co-occurrence structure is worth encoding explicitly.


In [ ]:
if annotated_mask.sum() >= 8:
    binary_gold = (gold > 0.5).astype(int)
    co = binary_gold.T.dot(binary_gold)

    plt.figure(figsize=(9, 7.5))
    sns.heatmap(co, annot=True, fmt="d", cmap=CMAP_SEQ, linewidths=0.5, linecolor="white")
    plt.title(f"Co-occurrence counts on the annotated subset (n={int(annotated_mask.sum())})")
    plt.tight_layout()
    plt.show()

    G = nx.Graph()
    for t in TARGETS:
        G.add_node(t, size=binary_gold[t].sum())
    for i, a in enumerate(TARGETS):
        for b in TARGETS[i+1:]:
            w = int(co.loc[a, b])
            if w > 0:
                G.add_edge(a, b, weight=w)

    pos = nx.spring_layout(G, seed=SEED, k=0.9)
    sizes = [G.nodes[n]["size"] * 40 + 200 for n in G.nodes]
    weights = [G[u][v]["weight"] for u, v in G.edges]

    plt.figure(figsize=(9, 8))
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=[FIRE]*len(G.nodes), alpha=0.85)
    nx.draw_networkx_edges(G, pos, width=[w / max(weights, default=1) * 5 for w in weights],
                            alpha=0.35, edge_color=MIDLINE)
    nx.draw_networkx_labels(G, pos, font_size=9)
    plt.title("Co-occurrence network (node size = positive count, edge width = shared studies)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Fewer than 8 annotated studies — co-occurrence analysis would be noise.")


## 8. Self-checks

Small assertions on the numbers this notebook computed, so a re-run on an updated dataset fails loudly instead of silently drifting.


In [ ]:
assert n_annot <= n_total, "annotated count exceeds total — annotation mask is wrong"
assert set(TARGETS).issubset(train_df.columns), "a target column is missing from train.csv"
if len(hdr_df) and "tag_side" in hdr_df.columns:
    assert hdr_df["tag_side"].isin(["L", "R", None]).all(), "unexpected laterality tag value"
if annotated_mask.sum() >= 8:
    assert (co.values.diagonal() >= 0).all(), "co-occurrence diagonal should be non-negative"

print("self-checks passed")


## 9. Summary — what this means for modeling

- **Annotated subset is small.** Any per-target AUC computed on it alone (see §3) has a wide Wilson/Hanley–McNeil interval — enough to treat small differences between approaches as noise, not signal. This is the argument for deriving auxiliary targets from `Report` text rather than training on the annotated subset alone.
- **Reports carry real signal, and `test.csv` has none.** The schema itself rules out a text-branch fusion model at inference; the only admissible uses are as training targets, as a distilled auxiliary signal, or as a per-study confidence weight.
- **Shared-report groups need to be respected in any split.** Splitting a template-report group across folds scores a model against a target it effectively trained on elsewhere.
- **Pixel spacing varies enough that a fixed-pixel resize is not neutral.** A physical-scale crop (fixed mm before resizing) removes a nuisance transform the encoder would otherwise have to learn to ignore.
- **The laterality tag covers roughly half the corpus** (update with your actual number from §6) — check whether the x-coordinate fallback's agreement is high enough to trust before using it to fill the rest, and never blind-trust it.
- **Compartment/side targets (Medial/Lateral pairs) are only meaningful once laterality is normalised** — medial and lateral are defined relative to the body's midline, not the image frame.
- **Co-occurrence is non-trivial** (§7) — worth knowing before deciding whether twelve independent heads or a shared representation is the better inductive bias.

---
*Numbers in italics above should be replaced with your actual run's figures once this notebook executes on the real mount — the structure and reasoning don't change, only the constants.*
